In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [143]:
import h5py
import os
import tqdm
from pathlib import Path

def slices(num_temps, num_splits):
    stride = int(num_temps/num_splits)
    start, stop = 0, stride + 1
    while stop <= num_temps:
        start += stride
        stop += stride
        yield slice(start, stop)

def splitbank(inputfilename, outdir):
    Path(outdir).mkdir(parents=True, exist_ok=True)
    outfile = os.path.join(outdir, Path(inputfilename).stem + "_SPLITBANK_{}.hdf")

    with h5py.File(inputfilename, "r") as h5in:
        num_temps = len((list(h5in.values())[0]))
        attributes = dict(h5in.attrs)
        for i, s in tqdm.tqdm(enumerate((slices(num_temps, num_splits))), total=num_splits):
            with h5py.File(outfile.format(i), "w") as h5out:
                h5out.attrs.update(attributes)
                for k, v in h5in.items():
                    assert isinstance(v, h5py.Dataset), "Can only copy Datasets."
                    h5out[k] = v[s]

num_splits = 100
outdir = "./splitbank"
inputfile = "../bank/full-correct-tau-xhm-spinning-2filter-listharms.hdf"
splitbank(inputfile, outdir)

100%|████████████████████████████████████████| 100/100 [00:00<00:00, 228.75it/s]
